In [1]:
import sys
import os
import pandas as pd
import numpy as np
from pycox.models.cox import CoxPH
from pycox.evaluation import EvalSurv
import matplotlib.pyplot as plt
import torchtuples as tt
sys.path.append(os.path.abspath("../../"))
from src.dataset.generate_dataset import TorchPreprocessing
from src.dataset.DataSet import SurvivalDataSet
from src.utils.Preprocessing import Preprocessor
from src.utils.ConvertTextToCsv import TextToCsv
from src.dataset.split_data import split_data_Train_Val_Test, create_dataloaders_train_val_test
import scipy.integrate
from sklearn.preprocessing import StandardScaler
from src.utils.cox_models import *
scipy.integrate.simps = scipy.integrate.simpson
from sksurv.nonparametric import kaplan_meier_estimator
from sksurv.compare import compare_survival
from sksurv.util import Surv
from itertools import product
import torch

%load_ext autoreload
%autoreload 2
%matplotlib inline

In [2]:
pp = Preprocessor()

In [3]:
df_clinical_data = pd.read_csv("../../data/raw/brca_tcga_pub2015_clinical_data.tsv", sep='\t')
df_clinical_data = pp.clean_columns_dataset(df_clinical_data)
list_df = pp.total_type_len_type_cancer(df_clinical_data)
df_clinical_data["Tumor-Cancer"] = list_df
df_clinical_data["Tumor-Cancer"].unique()

df_mRNA_raw_data = TextToCsv("../../data/raw/data_mrna_seq_v2_rsem.txt")
clean_mRNA_df = pp.eliminate_zero_genes(df_mRNA_raw_data, "Hugo_Symbol")


Luminal A: 330 - Total(%): 0.40
Luminal B: 81 - Total(%):0.10
HER2-enriched: 23 - Total(%):0.03
TNBC: 85 - Total(%)0.10 
UNK: 298 - Total(%) 0.36
Shape of the CSV: (20440, 819)
Genes before Treshold: 818
count     818.000000
mean     2911.273839
std       351.415192
min         0.000000
25%      2709.250000
50%      2866.500000
75%      3078.500000
max      4575.000000
dtype: float64
Threshold (>80% zeros): 16352 samples
After the treshold: 818


In [ ]:
df_mrna_clean = pp.eliminate_zero_genes(df_mRNA_raw_data, "Hugo_Symbol")
df_merged = TorchPreprocessing(clean_mRNA_df,df_clinical_data, 20000).get_comparation_df()
 
comparation_df = df_merged.loc[
    df_merged["Tumor-Cancer"].isin(["Luminal A", "Luminal B", "TNBC", "HER2-enriched"]),
]

comparation_df["Tumor-Cancer"].unique()

Genes before Treshold: 818
count     818.000000
mean     2911.273839
std       351.415192
min         0.000000
25%      2709.250000
50%      2866.500000
75%      3078.500000
max      4575.000000
dtype: float64
Threshold (>80% zeros): 16352 samples
After the treshold: 818
Genes before Treshold: 2003
count    2003.000000
mean       76.306041
std       158.918820
min         0.000000
25%         0.000000
50%         0.000000
75%        20.000000
max       519.000000
dtype: float64
Threshold (>80% zeros): 415 samples
After the treshold: 1805


array(['Luminal A', 'TNBC', 'Luminal B', 'HER2-enriched'], dtype=object)

In [5]:
zero_reduced_df = pp.eliminate_zero_genes(comparation_df, "Tumor-Cancer")
print(f"Samples: {zero_reduced_df.shape[0]}, Genes: {zero_reduced_df.shape[1]}")
zero_reduced_df =  zero_reduced_df.drop(["Sample ID"], axis=1)
results_df, desing, expr = pp.initialize_limma(zero_reduced_df, column="Tumor-Cancer", column_event="Overall Survival (Months)", column_status="Overall Survival Status")

Genes before Treshold: 1805
count    1805.000000
mean       30.929640
std        84.082264
min         0.000000
25%         0.000000
50%         0.000000
75%         2.000000
max       415.000000
dtype: float64
Threshold (>80% zeros): 415 samples
After the treshold: 1805
Samples: 519, Genes: 1806


In [6]:
results_df

,HER2-enriched,Luminal A,Luminal B,TNBC,AveExpr,F,pvalue,adj_pvalue
YTHDF1,10.879097,10.704586,10.782180,10.716942,10.726453,69262.336815,0.000000e+00,0.000000e+00
CCR7,6.044679,5.968886,6.192913,6.938782,6.166055,1634.501163,0.000000e+00,0.000000e+00
FBXO48,5.023124,4.763917,4.780206,4.962661,4.810496,6907.552264,0.000000e+00,0.000000e+00
PTPRN2,6.436115,7.635121,7.006290,5.565086,7.144821,1749.138908,0.000000e+00,0.000000e+00
TRIOBP,10.290598,10.180434,10.190821,10.256202,10.199346,42629.167766,0.000000e+00,0.000000e+00
...,...,...,...,...,...,...,...,...
SSX4,0.762482,0.202256,0.332518,0.421870,0.283380,15.974664,4.389646e-13,4.399411e-13
RPTN,0.281796,0.270517,0.264407,0.518887,0.310740,15.355910,1.456288e-12,1.458717e-12
REXO1L1P,0.128429,0.201275,0.248773,0.140834,0.195561,15.321054,1.557997e-12,1.559728e-12
CSN1S1,0.152372,0.442308,0.377112,0.388976,0.410550,14.655928,5.644621e-12,5.647756e-12


In [ ]:
N_GENES = 20000
top_genes_limma = results_df.sort_values("pvalue").index[:N_GENES]
Torch_preprocessing = TorchPreprocessing(df_mrna_clean, df_clinical_data, N_GENES)
Torch_preprocessing.genes_expression = top_genes_limma

In [8]:
Torch_preprocessing = TorchPreprocessing(df_mrna_clean, df_clinical_data, N_GENES)
Torch_preprocessing.genes_expression = top_genes_limma
X_scaled, durations, events, scaler = Torch_preprocessing.get_data_set(time_months=60)

Genes before Treshold: 103
count    103.000000
mean       1.368932
std        3.845013
min        0.000000
25%        0.000000
50%        0.000000
75%        0.000000
max       22.000000
dtype: float64
Threshold (>80% zeros): 415 samples
After the treshold: 103


In [9]:
X_scaled

array([[-0.74483652,  0.62116313,  1.05343255, ...,  0.13532206,
         0.97380478,  0.34969113],
       [ 4.15966731,  1.28554919,  1.00462229, ...,  1.00813543,
        -0.24730809, -2.03918366],
       [ 0.81298507,  1.29049904, -1.11227861, ..., -1.31516881,
        -1.2025222 ,  0.96894372],
       ...,
       [ 0.6350017 ,  1.67397423, -1.61943363, ..., -1.54581102,
        -3.19078758, -0.83916646],
       [-0.41089421, -0.25876731, -0.23212033, ..., -0.92968234,
         0.16585446, -0.01342517],
       [-1.11172281, -0.54302985, -0.23070348, ...,  0.03345942,
         0.24874648,  0.37402695]])